## Part 1. Load parking space data

In [15]:
import pandas as pd
import numpy as np
import json

df = pd.read_csv("../data/mtl_parking.csv", encoding="cp1252")
print(df.shape)
df.head()
df.columns.tolist()


(19640, 15)


['sNoPlace',
 'nLongitude',
 'nLatitude',
 'nPositionCentreLongitude',
 'nPositionCentreLatitude',
 'sStatut',
 'sGenre',
 'sType',
 'sAutreTete',
 'sNomRue',
 'nSupVelo',
 'sTypeExploitation',
 'nTarifHoraire',
 'sLocalisation',
 'nTarifMax']

## Part 2. Clean the data
Potential parking space candidates for conversion only include off-street parking (parking lots), so I need to keep every row that is off-street ("H" under the Parking Location column)

Checking how many rows under "S" and "H" parking lcoation. "S" = sur rue (on street parking), "H" = hors rue (off street)

In [16]:
df["sLocalisation"].value_counts().reindex(["S", "H"], fill_value=0)

sLocalisation
S    17962
H     1678
Name: count, dtype: int64

In [17]:
df_h = df[df["sLocalisation"] == "H"] # this keeps only off-street parking spaces

## Join the parking points to their borough
Using a Montreal boundaries GeoJson file

Check which column is associated with the borough names after reading the boundaries dataset:

In [18]:
import pandas as pd
import geopandas as gpd

In [19]:
boroughs = gpd.read_file("../data/boundaries.geojson")
print(boroughs.columns.tolist())

['CODEID', 'NOM', 'NOM_OFFICIEL', 'CODEMAMH', 'CODE_3C', 'NUM', 'ABREV', 'TYPE', 'COMMENT', 'DATEMODIF', 'geometry']


In [20]:
# Create points from the filtered rows (in df_h)
points_h = gpd.GeoDataFrame(
    df_h,
    geometry=gpd.points_from_xy(df_h["nLongitude"], df_h["nLatitude"]),
    crs="EPSG:4326"
)

# Ensure both layers use the same coordinate reference system (CRS)
boroughs = boroughs.to_crs(points_h.crs)

# Assign a borough to each H point
points_h_b = gpd.sjoin(
    points_h,
    boroughs[["NOM", "geometry"]],
    how="left",
    predicate="within"
)

## Part 3. Write out the rows as GeoJSON features and export into parking.geojson

In [21]:
def row_to_feature(row):
    """Convert one parking-table row into a GeoJSON feature."""
    return {
        "type": "Feature",
        "geometry": {
            "type": "Point",
            "coordinates": [
                float(row["nLongitude"]),
                float(row["nLatitude"])
            ]
        },
        "properties": {
            "Borough": row["NOM"],   # renaming
            "Hourly rate": row["nTarifHoraire"],
            "Street Name": row["sNomRue"]
        }
    }

In [22]:
features = []
for _, row in points_h_b.iterrows():
    features.append(row_to_feature(row))  # add row_to_feature(row) to the list

collection = {"type": "FeatureCollection", "features": features}

out_path = "../data/parking.geojson"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(collection, f, separators=(",", ":"), allow_nan=False)
    # separators: no spaces, a smaller file.
    # allow_nan=False: stop with an error if any value is NaN (an empty cell).
    # Browsers cannot read NaN in JSON, so without this line the file writes
    # fine and the map fails later with a confusing message. Fail here instead.

import os
print(len(features), "features written to", out_path)
print(len(points_h_b), "features in the original table")
print(round(os.path.getsize(out_path) / 1_000_000, 1), "MB")

1678 features written to ../data/parking.geojson
1678 features in the original table
0.4 MB


## Part 5. Check cell

In [23]:
with open("../data/parking.geojson", encoding="utf-8") as f:
    back = json.load(f)

problems = []
feats = back.get("features", [])
if back.get("type") != "FeatureCollection":
    problems.append("The file is not a FeatureCollection.")
if len(feats) != len(points_h_b):
    problems.append(f"The file has {len(feats)} features but the DataFrame has {len(points_h_b)} rows.")
lons = [ft["geometry"]["coordinates"][0] for ft in feats]
lats = [ft["geometry"]["coordinates"][1] for ft in feats]
if feats and not all(-74.0 < x < -73.4 for x in lons):
    problems.append("Some longitudes are outside the island of Montreal. Are longitude and latitude swapped?")
if feats and not all(45.3 < y < 45.8 for y in lats):
    problems.append("Some latitudes are outside the island of Montreal. Are longitude and latitude swapped?")
if feats and not all({"Street Name", "Borough", "Hourly rate"} <= set(ft["properties"]) for ft in feats):
    problems.append("Some features are missing Street Name, Borough, or Hourly rate in their properties.")

print("All checks passed" if not problems else "Not yet:\n- " + "\n- ".join(problems))

All checks passed


Checking if any points have not been matched to a borough and how many boroughs have points

In [24]:
# Number of points matched to each borough
print(points_h_b["NOM"].value_counts(dropna=False))

# Number of distinct boroughs with points
print("Boroughs with points:", points_h_b["NOM"].nunique())

# Number of boroughs defined in the boundaries file
print("Boroughs in boundaries file:", boroughs["NOM"].nunique())

# Points that did not match a borough
print("Unmatched points:", points_h_b["NOM"].isna().sum())

NOM
Verdun                                 780
Rosemont-La Petite-Patrie              450
Outremont                              169
Ville-Marie                            130
Le Plateau-Mont-Royal                   79
Le Sud-Ouest                            39
Ahuntsic-Cartierville                   21
Côte-des-Neiges-Notre-Dame-de-Grâce     10
Name: count, dtype: int64
Boroughs with points: 8
Boroughs in boundaries file: 34
Unmatched points: 0
